# Vehicle Insurance Cross-Sell Response Propensity — Tier 0 Experiments

## Notebook purpose and status

This notebook is a clean Tier 0 experiment for the vehicle-insurance cross-sell response-propensity project. It is intentionally separate from the original `experimentation.ipynb`, which is preserved as a historical baseline.

The purpose of this notebook is to establish a trustworthy evaluation protocol before adding advanced feature engineering, resampling, or additional model families. Results from this notebook should be reproducible from top to bottom and should not depend on hidden state from the previous notebook.

**Current phase:** Tier 0 — evaluation correctness and business-aligned baseline measurement.

**Important status note:** this notebook is an experimental research artifact. It is not yet the final training pipeline or the production serving implementation.

## 1. Project objective

The project estimates which existing customers are most likely to respond positively to a vehicle-insurance cross-sell offer. The model receives customer information available before a marketing campaign and produces a response-propensity score. Customers are then ranked from highest to lowest score so that a campaign team can prioritize a selected top-K group.

The target is `Response`:

- `Response = 1`: the customer responded positively to the vehicle-insurance offer.
- `Response = 0`: the customer did not respond positively to the vehicle-insurance offer.

The current Phase 1 business action is **top-K customer selection**. The company may select K as a percentage of the scored population or as a fixed number of customers. This notebook evaluates top 1%, 5%, 10%, and 20% capacities, with top 10% as the initial illustrative operating point. Customers outside the selected group are lower-priority campaign cases; they are not rejected, denied coverage, or declared ineligible.

This model is a marketing-prioritization model only. It must not be used for claims approval, premium pricing, underwriting, fraud detection, insurance eligibility, policy cancellation, or automatic adverse decisions. The dataset supports response-propensity modeling, not causal claims about whether contacting a customer created an incremental response.

## 2. Evaluation contract

The primary question is:

> Does the model rank likely responders near the top well enough to improve the quality of a fixed-capacity marketing selection?

### Primary metrics

The primary metrics are ranking and top-K selection metrics:

- **PR-AUC / Average Precision:** measures ranking quality with greater focus on the minority positive class.
- **Precision@K:** among the customers selected in the top K%, how many actually responded?
- **Recall@K:** among all actual responders, how many were captured in the top K%?
- **Lift@K:** how much better the top-K selection performs compared with selecting customers at random.

Calculate these metrics at top 1%, 5%, 10%, and 20%. The operating point must be selected using validation data, not the final test set.

### Secondary metrics

The notebook may also report ROC-AUC, precision, recall, F1, and a confusion matrix. These are secondary because F1 and confusion-matrix metrics depend on a classification threshold, while the campaign decision is primarily based on ranking. The default threshold of 0.5 must not be treated as automatically correct.

### Data split and leakage rules

Use a stratified three-way split with training, validation, and one final untouched test set. The current recommended protocol is 80% training, 10% validation, and 10% final test, with a fixed random seed of 42 and cross-validation performed only inside the training portion when hyperparameters are tuned.

- Fit preprocessing only on training data.
- Fit models and perform hyperparameter search using training data.
- Use validation data for model comparisons, top-K operating-point decisions, and threshold analysis.
- Keep the final test set untouched until the model and evaluation protocol are frozen.
- Evaluate the final selected model on the final test set once for the final report.
- Never select a model because it produced the best final-test score.
- Compare all candidate models using the same split, preprocessing rules, and metric definitions.

### Baseline and model-selection rules

The Tier 0 comparison should include a simple baseline, an interpretable baseline where possible, and the current no-resampling LightGBM baseline. The purpose is to establish whether the model provides useful ranking skill above simple alternatives before introducing more complex changes.

A candidate should be judged primarily on PR-AUC and top-K selection quality, especially the agreed top-K capacities. Threshold-based metrics can inform the serving decision, but they must not replace the ranking evaluation.

## 3. Tier 0 notebook workflow

The notebook should follow this order:

1. Document the project objective and evaluation contract.
2. Load the expected dataset using a portable path.
3. Run dataset integrity checks and record the dataset snapshot.
4. Create the frozen train/validation/final-test split.
5. Build the baseline preprocessing and modeling flow.
6. Generate validation probabilities and calculate PR-AUC and top-K metrics.
7. Compare the baseline candidates using validation data only.
8. Perform threshold analysis as a secondary evaluation.
9. Freeze the final candidate and operating point.
10. Evaluate the frozen candidate once on the final test set.
11. Summarize findings, limitations, and the decision about whether Tier 1 is justified.

Do not move to advanced feature engineering or resampling until the Tier 0 baseline and evaluation protocol are working end to end.

## Notebook dependencies

In [1]:
import os
import warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

from imblearn.over_sampling import SMOTE

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    average_precision_score,
    auc,
    classification_report,
    roc_auc_score
)

import lightgbm as lgb
from lightgbm import LGBMClassifier
import mlflow
import pickle

warnings.filterwarnings("ignore")

## 4. Dataset loading and integrity checks

This section establishes that the notebook is using the expected dataset and that the data is safe to use for modeling. It is not intended to repeat every exploratory chart from the original notebook. The checks here are reproducibility and data-integrity gates that should run whenever this notebook is executed.

### Dataset-loading responsibilities

The loading step should document the expected file location, load the dataset into a dataframe, and record the dataset shape and columns. The path should be portable rather than tied to one absolute directory on one computer. The notebook should make it clear which data snapshot produced the experiment results.

### Integrity checks to complete

The code cells below should verify the following items before feature preparation or splitting:

1. **Expected schema:** confirm that the expected feature columns and the `Response` target exist.
2. **Unexpected columns:** identify columns that are not part of the documented data contract.
3. **Target validity:** confirm that `Response` contains only 0 and 1 and record the positive-class prevalence.
4. **Missing values:** report missing values by column and confirm whether any handling is required.
5. **Duplicate records:** check duplicate IDs and complete duplicate rows.
6. **Identifier handling:** confirm that `id` is unique, is not used as a predictive feature, and can be retained separately for future ranked-customer output.
7. **Data types:** confirm that numeric columns are numeric and that categorical-looking columns such as `Gender`, `Vehicle_Age`, `Vehicle_Damage`, `Region_Code`, and `Policy_Sales_Channel` are treated according to their meaning rather than only their storage dtype.
8. **Value ranges:** check that fields such as `Age`, `Annual_Premium`, `Vintage`, `Driving_License`, and `Previously_Insured` contain plausible values.
9. **Leakage review:** confirm that no target, post-campaign, or future-information column is included among the features.
10. **Dataset limitations:** record that the current dataset has no reliable campaign timestamp, is treated as one row per customer, and does not contain treatment/control information for causal or uplift modeling.

### Expected output

At the end of this section, the notebook should provide a concise dataset summary containing the row count, column list, target counts and prevalence, missing-value summary, duplicate summary, identifier check, and any accepted limitations. A reviewer should be able to understand what data entered the experiment without opening the old notebook.

In [2]:
from pathlib import Path

# Support execution from either the repository root or the notebooks directory.
current_dir = Path.cwd()
data_path = (current_dir / "data.csv") if current_dir.name == "notebooks" else (current_dir / "notebooks" / "data.csv")
data_path = data_path.resolve()

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at: {data_path}")

df = pd.read_csv(data_path)
print(f"Loaded dataset from: {data_path}")
print(f"Dataset shape: {df.shape}")
df.head()

Loaded dataset from: D:\projVehicleInsurance\vehicle-Insurance\notebooks\data.csv
Dataset shape: (381109, 12)


,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,1,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1
1,2,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0
2,3,Male,47,1,28.0,0,> 2 Years,Yes,38294.0,26.0,27,1
3,4,Male,21,1,11.0,1,< 1 Year,No,28619.0,152.0,203,0
4,5,Female,29,1,41.0,1,< 1 Year,No,27496.0,152.0,39,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 381109 entries, 0 to 381108
Data columns (total 12 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    381109 non-null  int64  
 1   Gender                381109 non-null  object 
 2   Age                   381109 non-null  int64  
 3   Driving_License       381109 non-null  int64  
 4   Region_Code           381109 non-null  float64
 5   Previously_Insured    381109 non-null  int64  
 6   Vehicle_Age           381109 non-null  object 
 7   Vehicle_Damage        381109 non-null  object 
 8   Annual_Premium        381109 non-null  float64
 9   Policy_Sales_Channel  381109 non-null  float64
 10  Vintage               381109 non-null  int64  
 11  Response              381109 non-null  int64  
dtypes: float64(3), int64(6), object(3)
memory usage: 34.9+ MB


confirms that no unexpected columns exist and schema is as expected. 

In [4]:
response_counts = df['Response'].value_counts()
print(f"Response counts:\n{response_counts}\n")

Response counts:
Response
0    334399
1     46710
Name: count, dtype: int64



In [5]:
positive_percentage = (response_counts[1] / response_counts.sum()) * 100
print(f"Percentage of positive responses: {positive_percentage:.2f}%")

Percentage of positive responses: 12.26%


The response counts confirm that response contains only 0 and 1 and the positive-class prevalence is 12.26%.


In [6]:
#Lets now check for missing values in each column
df.isnull().sum()

id                      0
Gender                  0
Age                     0
Driving_License         0
Region_Code             0
Previously_Insured      0
Vehicle_Age             0
Vehicle_Damage          0
Annual_Premium          0
Policy_Sales_Channel    0
Vintage                 0
Response                0
dtype: int64

Thus no missing values exist in any column.

In [7]:
#lets now check for duplicate ids and rows in the dataset
duplicate_ids = df.duplicated(subset='id').sum()
print(f"Number of duplicate IDs: {duplicate_ids}")

duplicate_rows = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_rows}")

Number of duplicate IDs: 0
Number of duplicate rows: 0


So no duplicate IDs or rows exist in the dataset.

In [8]:
#check for categorical columns and their unique values
categorical_columns = df.select_dtypes(include=['object']).columns

for col in categorical_columns:
    print(f"\nUnique values in column '{col}':")
    print(df[col].unique())


Unique values in column 'Gender':
['Male' 'Female']

Unique values in column 'Vehicle_Age':
['> 2 Years' '1-2 Year' '< 1 Year']

Unique values in column 'Vehicle_Damage':
['Yes' 'No']


In [9]:
# Check documented value ranges and binary domains.
range_violations = {
    "Age_non_positive": int((df["Age"] <= 0).sum()),
    "Annual_Premium_non_positive": int((df["Annual_Premium"] <= 0).sum()),
    "Vintage_negative": int((df["Vintage"] < 0).sum()),
    "Driving_License_invalid": int((~df["Driving_License"].isin([0, 1])).sum()),
    "Previously_Insured_invalid": int((~df["Previously_Insured"].isin([0, 1])).sum()),
}
range_violations = {name: count for name, count in range_violations.items() if count > 0}
if range_violations:
    raise ValueError(f"Value-range violations found: {range_violations}")


In [10]:
print("Dataset integrity checks passed.")

Dataset integrity checks passed.


## 5. Data Preparation

Lets prepare the data now

In [11]:
random_state = 42

In [12]:
X = df.drop(columns=["Response", "id"])
y = df["Response"]
id_col = df["id"]
print(f"Features shape: {X.shape}, Target shape: {y.shape}")

Features shape: (381109, 10), Target shape: (381109,)


In [13]:
# LETS PREPARE THE TRAIN,TEST AND VALIDATION DATASETS
#first lets separate train  and rest in 80:20 ratio
X_train, X_rest, y_train, y_rest, id_train, id_rest = train_test_split(X, y, id_col, test_size=0.2, random_state=random_state, stratify=y)

#now divide rest into test and validation in 50:50 ratio
X_test, X_val, y_test, y_val, id_test, id_val = train_test_split(X_rest, y_rest, id_rest, test_size=0.5, random_state=random_state, stratify=y_rest)


In [14]:
print("Training set:")
print(X_train.shape, y_train.shape)

print("\nValidation set:")
print(X_val.shape, y_val.shape)

print("\nTest set:")
print(X_test.shape, y_test.shape)

Training set:
(304887, 10) (304887,)

Validation set:
(38111, 10) (38111,)

Test set:
(38111, 10) (38111,)


In [15]:
# Verify target prevalence, ID separation, and complete dataset coverage.
split_summary = pd.DataFrame({
    "rows": [len(y_train), len(y_val), len(y_test)],
    "positive_responses": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
    "positive_rate": [y_train.mean(), y_val.mean(), y_test.mean()],
}, index=["train", "validation", "test"])

print("Target prevalence by split:")
display(split_summary.style.format({"positive_rate": "{:.4%}"}))

# Every pair of subsets must contain different customer IDs.
train_ids = set(id_train)
validation_ids = set(id_val)
test_ids = set(id_test)

overlap_counts = {
    "train_validation": len(train_ids & validation_ids),
    "train_test": len(train_ids & test_ids),
    "validation_test": len(validation_ids & test_ids),
}

if any(overlap_counts.values()):
    raise ValueError(f"Customer ID overlap detected: {overlap_counts}")

# The split must cover every original row exactly once.
combined_ids = train_ids | validation_ids | test_ids
expected_ids = set(id_col)
missing_ids = expected_ids - combined_ids
unexpected_ids = combined_ids - expected_ids
total_split_rows = len(y_train) + len(y_val) + len(y_test)

if missing_ids or unexpected_ids or total_split_rows != len(df):
    raise ValueError(
        "Dataset coverage check failed: "
        f"missing_ids={len(missing_ids)}, "
        f"unexpected_ids={len(unexpected_ids)}, "
        f"split_rows={total_split_rows}, original_rows={len(df)}"
    )

print("ID overlap check passed: no customer appears in more than one split.")
print("Coverage check passed: every original customer appears exactly once across the splits.")

Target prevalence by split:


,rows,positive_responses,positive_rate
train,304887,37368,12.2563%
validation,38111,4671,12.2563%
test,38111,4671,12.2563%


ID overlap check passed: no customer appears in more than one split.
Coverage check passed: every original customer appears exactly once across the splits.


## 6. Baseline Preprocessing and Modeling Flow

### 6.1 Preprocessing

In [16]:
# lets firts define the feature types
categorical_features = [
    "Gender",
    "Vehicle_Age",
    "Vehicle_Damage",
    "Region_Code",
    "Policy_Sales_Channel",
]
numerical_features = ["Age", "Annual_Premium", "Vintage"]
binary_features = ["Driving_License", "Previously_Insured"]

In [17]:
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numerical", "passthrough", numerical_features),
        ("binary", "passthrough", binary_features)
    ]
)

In [18]:
preprocessor.fit(X_train)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numerical', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fea

In [19]:
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)
X_val_processed = preprocessor.transform(X_val)

Lets check the preprocessed data

In [20]:
print("Train columns:", X_train_processed.shape[1])
print("Validation columns:", X_val_processed.shape[1])
print("Test columns:", X_test_processed.shape[1])

Train columns: 219
Validation columns: 219
Test columns: 219


In [21]:
feature_names = preprocessor.get_feature_names_out()

In [22]:
same_feature_structure = (
    X_train_processed.shape[1] == len(feature_names)
    and X_val_processed.shape[1] == len(feature_names)
    and X_test_processed.shape[1] == len(feature_names)
)
print("Same feature structure:", same_feature_structure)

Same feature structure: True


In [23]:
numerical_present = all(
    f"numerical__{col}" in feature_names
    for col in numerical_features
)

# 4. Binary features included
binary_present = all(
    f"binary__{col}" in feature_names
    for col in binary_features
)


print("All numerical features included:", numerical_present)
print("All binary features included:", binary_present)

All numerical features included: True
All binary features included: True


In [24]:
target_present = any("Response" in col for col in feature_names)

id_present = any(
    col == "id" or col.endswith("__id")
    for col in feature_names
)
print("Response present:", target_present)
print("ID present:", id_present)

Response present: False
ID present: False


### 6.2 Define metrics

In [25]:
def precision_at_k(y_true , y_proba , k):
    """
    Calculate precision at k% for binary classification.

    Parameters:
    y_true (array-like): True binary labels (0 or 1).
    y_proba (array-like): Predicted probabilities for the positive class.
    k (int): Percentage of top instances to consider.

    Returns:
    float: Precision at k.
    """
    # Ensure inputs are numpy arrays
    y_true = np.array(y_true)
    y_proba = np.array(y_proba)
    
    n_top = int(len(y_true) * (k / 100))
    
    y_top_k_indices = np.argsort(y_proba)[::-1][:n_top]
    top_k_true = y_true[y_top_k_indices]
    
    return top_k_true.sum() / n_top if n_top > 0 else 0.0

    

In [26]:
def recall_at_k(y_true , y_proba , k):
    """
    Calculate recall at k% for binary classification.

    Parameters:
    y_true (array-like): True binary labels (0 or 1).
    y_proba (array-like): Predicted probabilities for the positive class.
    k (int): Percentage of top instances to consider.

    Returns:
    float: Recall at k.
    """
    y_true = np.array(y_true)
    y_proba = np.array(y_proba)
    
    n_top = int(len(y_true) * (k / 100))
    top_k_indices = np.argsort(y_proba)[::-1][:n_top]
    top_k_true = y_true[top_k_indices]
    
    return top_k_true.sum() / y_true.sum() if y_true.sum() > 0 else 0.0

In [27]:
def Lift_at_k(y_true,y_proba,k):
    """
    Calculate lift at k% for binary classification.

    Parameters:
    y_true (array-like): True binary labels (0 or 1).
    y_proba (array-like): Predicted probabilities for the positive class.
    k (int): Percentage of top instances to consider.

    Returns:
    float: Lift at k.
    """
    y_true = np.array(y_true)
    y_proba = np.array(y_proba)
    
    precision_value = precision_at_k(y_true, y_proba, k)
    overall_positive_rate = np.mean(y_true)

    return precision_value / overall_positive_rate if overall_positive_rate > 0 else 0.0 

In [28]:
def pr_auc(y_true, y_proba):
    """
    Calculate the area under the precision-recall curve (PR AUC) for binary classification.

    Parameters:
    y_true (array-like): True binary labels (0 or 1).
    y_proba (array-like): Predicted probabilities for the positive class.

    Returns:
    float: Area under the precision-recall curve.
    """
    return average_precision_score(y_true, y_proba)

### 6.3 MLFLOW

Before Training lets initiate the mlflow

In [29]:
def metric_logger(y_true,y_pred, y_proba, k):
    """
    Log various evaluation metrics for binary classification.

    Parameters:
    y_true (array-like): True binary labels (0 or 1).
    y_pred (array-like): Predicted binary labels.
    y_proba (array-like): Predicted probabilities for the positive class.
    k (int): Percentage of top instances to consider for precision, recall, and lift.

    Returns:
    dict: Dictionary containing calculated metrics.
    """
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1_score": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_proba),
        
        "precision_at_k": precision_at_k(y_true, y_proba, k),
        "recall_at_k": recall_at_k(y_true, y_proba, k),
        "lift_at_k": Lift_at_k(y_true, y_proba, k),
        "pr_auc": pr_auc(y_true, y_proba)
    }
    mlflow.log_metrics(metrics)
    return metrics

In [30]:
from dotenv import load_dotenv
load_dotenv()  

True

In [31]:
tracking_uri = os.getenv("MLFLOW_TRACKING_URI")
if not tracking_uri:
    raise ValueError("MLFLOW_TRACKING_URI is not set in the environment variables.")
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("Vehicle_Insurance_Experimentation_v2")
print(f"Tracking URI: {tracking_uri}")
print("MLflow experiment configured.")

Tracking URI: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow
MLflow experiment configured.


### 6.4 Model Training

train with lightgbm.

In [32]:
#MLFLOW RUN
run_name = "run0_lgbm_no_resampling_baseline"
with mlflow.start_run(run_name=run_name) as run:
    print(f"MLflow run started with ID: {run.info.run_id}")
    mlflow.log_params(
        {
            "Model": "LightGBM",
            "Objective": "binary",
            "random_state": random_state,
            "resampling": "none",
            "class_weighting": "none",
            "split_proportions": "stratified_train: 80%, validation: 10%, test: 10%",
            "preprocessing": "one_hot_categorical_passthrough_numeric_binary",
        }
    )
    
    lgbm_baseline = LGBMClassifier(random_state=random_state,objective='binary')
    lgbm_baseline.fit(X_train_processed, y_train)

    y_val_pred = lgbm_baseline.predict(X_val_processed)
    y_val_proba = lgbm_baseline.predict_proba(X_val_processed)[:, 1]
    
    validation_metrics = metric_logger(
        y_true=y_val,
        y_pred=y_val_pred,
        y_proba=y_val_proba,
        k=10
    )
    print("Validation metrics:")
    print(validation_metrics)

    mlflow.lightgbm.log_model(
        lgbm_baseline,
        artifact_path="model"
    )

    mlflow.sklearn.log_model(
        preprocessor,
        artifact_path="preprocessor"
    )

    print(f"MLflow run ID: {run.info.run_id}")

MLflow run started with ID: c8ece3d857ff40f8bf860e3442bcf4d2
[LightGBM] [Info] Number of positive: 37368, number of negative: 267519
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010692 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 870
[LightGBM] [Info] Number of data points in the train set: 304887, number of used features: 155
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.122563 -> initscore=-1.968376
[LightGBM] [Info] Start training from score -1.968376


2026/08/22 18:16:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Validation metrics:
{'accuracy': 0.8771220907349584, 'precision': 0.3125, 'recall': 0.002140869192892314, 'f1_score': 0.00425260472039124, 'roc_auc': 0.8562721289305181, 'precision_at_k': np.float64(0.39149829441091577), 'recall_at_k': np.float64(0.3194176835795333), 'lift_at_k': np.float64(3.19426065045909), 'pr_auc': 0.36492881778490055}


2026/08/22 18:16:25 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/08/22 18:16:34 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/08/22 18:16:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/22 18:16:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For 

MLflow run ID: c8ece3d857ff40f8bf860e3442bcf4d2
🏃 View run run0_lgbm_no_resampling_baseline at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/1/runs/c8ece3d857ff40f8bf860e3442bcf4d2
🧪 View experiment at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/1


### 6.5 Top-K selection evaluation

The model is intended to rank customers for campaign outreach. Therefore, evaluate the quality of the highest-scored 1%, 5%, 10%, and 20% of validation customers. These percentages represent possible campaign capacities; they are evaluation scenarios, not fixed company requirements.

In [33]:
top_k_percentages = [1, 5, 10, 20]

top_k_results = []
for top_k in top_k_percentages:
    top_k_results.append({
        "selection_percentage": top_k,
        "customers_selected": int(len(y_val) * top_k / 100),
        "precision_at_k": precision_at_k(y_val, y_val_proba, top_k),
        "recall_at_k": recall_at_k(y_val, y_val_proba, top_k),
        "lift_at_k": Lift_at_k(y_val, y_val_proba, top_k),
    })

top_k_results = pd.DataFrame(top_k_results)
display(top_k_results.style.format({
    "precision_at_k": "{:.4%}",
    "recall_at_k": "{:.4%}",
    "lift_at_k": "{:.3f}",
}))

,selection_percentage,customers_selected,precision_at_k,recall_at_k,lift_at_k
0,1,381,46.1942%,3.7679%,3.769
1,5,1905,41.6273%,16.9771%,3.396
2,10,3811,39.1498%,31.9418%,3.194
3,20,7622,35.2401%,57.5037%,2.875


### 6.6 Simple targeting baselines

These baselines provide reference points. Predict-none represents contacting nobody, while predict-all represents treating every customer as a positive response. Their performance prevents the LightGBM result from being interpreted without context.

In [34]:
baseline_predictions = {
    "predict_none": np.zeros(len(y_val), dtype=int),
    "predict_all": np.ones(len(y_val), dtype=int),
}

baseline_probabilities = {
    "predict_none": np.zeros(len(y_val), dtype=float),
    "predict_all": np.ones(len(y_val), dtype=float),
}

baseline_rows = []
for baseline_name in baseline_predictions:
    baseline_y_pred = baseline_predictions[baseline_name]
    baseline_y_proba = baseline_probabilities[baseline_name]
    baseline_rows.append({
        "model": baseline_name,
        "accuracy": accuracy_score(y_val, baseline_y_pred),
        "precision": precision_score(y_val, baseline_y_pred, zero_division=0),
        "recall": recall_score(y_val, baseline_y_pred, zero_division=0),
        "f1_score": f1_score(y_val, baseline_y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_val, baseline_y_proba),
        "pr_auc": pr_auc(y_val, baseline_y_proba),
        "precision_at_10pct": precision_at_k(y_val, baseline_y_proba, 10),
        "recall_at_10pct": recall_at_k(y_val, baseline_y_proba, 10),
        "lift_at_10pct": Lift_at_k(y_val, baseline_y_proba, 10),
    })

baseline_results = pd.DataFrame(baseline_rows)
display(baseline_results.style.format({
    "accuracy": "{:.4%}",
    "precision": "{:.4%}",
    "recall": "{:.4%}",
    "f1_score": "{:.4%}",
    "roc_auc": "{:.4f}",
    "pr_auc": "{:.4f}",
    "precision_at_10pct": "{:.4%}",
    "recall_at_10pct": "{:.4%}",
    "lift_at_10pct": "{:.3f}",
}))

,model,accuracy,precision,recall,f1_score,roc_auc,pr_auc,precision_at_10pct,recall_at_10pct,lift_at_10pct
0,predict_none,87.7437%,0.0000%,0.0000%,0.0000%,0.5000,0.1226,11.9654%,9.7624%,0.976
1,predict_all,12.2563%,12.2563%,100.0000%,21.8363%,0.5000,0.1226,11.9654%,9.7624%,0.976


### 6.7 Ranking baseline comparison

This section compares LightGBM with an interpretable Logistic Regression model and a random-ranking baseline. Every candidate uses the same validation split and the same top-K metric definitions.

In [35]:
logistic_scaler = StandardScaler(with_mean=False)
X_train_logistic = logistic_scaler.fit_transform(X_train_processed)
X_val_logistic = logistic_scaler.transform(X_val_processed)

logistic_baseline = LogisticRegression(random_state=random_state, max_iter=1000)
logistic_baseline.fit(X_train_logistic, y_train)
y_logistic_proba = logistic_baseline.predict_proba(X_val_logistic)[:, 1]
y_logistic_pred = logistic_baseline.predict(X_val_logistic)

random_generator = np.random.default_rng(random_state)
y_random_proba = random_generator.random(len(y_val))
y_random_pred = (y_random_proba >= 0.5).astype(int)

model_scores = {
    "LightGBM": y_val_proba,
    "Logistic Regression": y_logistic_proba,
    "Random ranking": y_random_proba,
    "Predict none": np.zeros(len(y_val), dtype=float),
    "Predict all": np.ones(len(y_val), dtype=float),
}
model_predictions = {
    "LightGBM": y_val_pred,
    "Logistic Regression": y_logistic_pred,
    "Random ranking": y_random_pred,
    "Predict none": np.zeros(len(y_val), dtype=int),
    "Predict all": np.ones(len(y_val), dtype=int),
}

comparison_rows = []
for model_name, model_proba in model_scores.items():
    model_pred = model_predictions[model_name]
    for top_k in top_k_percentages:
        comparison_rows.append({
            "model": model_name,
            "selection_percentage": top_k,
            "accuracy": accuracy_score(y_val, model_pred),
            "precision": precision_score(y_val, model_pred, zero_division=0),
            "recall": recall_score(y_val, model_pred, zero_division=0),
            "f1_score": f1_score(y_val, model_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_val, model_proba),
            "pr_auc": pr_auc(y_val, model_proba),
            "precision_at_k": precision_at_k(y_val, model_proba, top_k),
            "recall_at_k": recall_at_k(y_val, model_proba, top_k),
            "lift_at_k": Lift_at_k(y_val, model_proba, top_k),
        })

comparison_results = pd.DataFrame(comparison_rows)
display(comparison_results.style.format({
    "accuracy": "{:.4%}", "precision": "{:.4%}",
    "recall": "{:.4%}", "f1_score": "{:.4%}",
    "roc_auc": "{:.4f}", "pr_auc": "{:.4f}",
    "precision_at_k": "{:.4%}", "recall_at_k": "{:.4%}",
    "lift_at_k": "{:.3f}",
}))

,model,selection_percentage,accuracy,precision,recall,f1_score,roc_auc,pr_auc,precision_at_k,recall_at_k,lift_at_k
0,LightGBM,1,87.7122%,31.2500%,0.2141%,0.4253%,0.8563,0.3649,46.1942%,3.7679%,3.769
1,LightGBM,5,87.7122%,31.2500%,0.2141%,0.4253%,0.8563,0.3649,41.6273%,16.9771%,3.396
2,LightGBM,10,87.7122%,31.2500%,0.2141%,0.4253%,0.8563,0.3649,39.1498%,31.9418%,3.194
3,LightGBM,20,87.7122%,31.2500%,0.2141%,0.4253%,0.8563,0.3649,35.2401%,57.5037%,2.875
4,Logistic Regression,1,87.7148%,33.3333%,0.2355%,0.4677%,0.8494,0.3401,35.6955%,2.9116%,2.912
5,Logistic Regression,5,87.7148%,33.3333%,0.2355%,0.4677%,0.8494,0.3401,36.8504%,15.0289%,3.007
6,Logistic Regression,10,87.7148%,33.3333%,0.2355%,0.4677%,0.8494,0.3401,36.9194%,30.1220%,3.012
7,Logistic Regression,20,87.7148%,33.3333%,0.2355%,0.4677%,0.8494,0.3401,34.2430%,55.8767%,2.794
8,Random ranking,1,49.4582%,11.9372%,48.9831%,19.1962%,0.4883,0.1183,11.2861%,0.9206%,0.921
9,Random ranking,5,49.4582%,11.9372%,48.9831%,19.1962%,0.4883,0.1183,11.3911%,4.6457%,0.929


### 6.8 LightGBM threshold analysis

The default threshold of 0.5 is not assumed to be correct. This analysis shows how LightGBM precision, recall, F1, and contact rate change across candidate thresholds. It is descriptive only; do not freeze a threshold until the business operating objective is selected.

In [36]:
threshold_rows = []
for threshold in np.arange(0.05, 1.00, 0.05):
    threshold_pred = (y_val_proba >= threshold).astype(int)
    threshold_rows.append({
        "threshold": round(float(threshold), 2),
        "contact_rate": threshold_pred.mean(),
        "precision": precision_score(y_val, threshold_pred, zero_division=0),
        "recall": recall_score(y_val, threshold_pred, zero_division=0),
        "f1_score": f1_score(y_val, threshold_pred, zero_division=0),
    })

threshold_results = pd.DataFrame(threshold_rows)
display(threshold_results.style.format({
    "threshold": "{:.2f}", "contact_rate": "{:.4%}",
    "precision": "{:.4%}", "recall": "{:.4%}",
    "f1_score": "{:.4%}",
}))

,threshold,contact_rate,precision,recall,f1_score
0,0.05,47.7736%,25.1936%,98.2017%,40.0997%
1,0.10,43.0716%,27.0423%,95.0332%,42.1038%
2,0.15,37.9680%,29.0947%,90.1306%,43.9893%
3,0.20,31.7284%,31.6159%,81.8454%,45.6124%
4,0.25,25.9663%,33.3872%,70.7343%,45.3628%
5,0.30,17.7613%,36.1058%,52.3228%,42.7273%
6,0.35,9.6586%,39.3100%,30.9784%,34.6504%
7,0.40,3.3193%,42.8458%,11.6035%,18.2615%
8,0.45,0.6927%,46.5909%,2.6333%,4.9848%
9,0.50,0.0840%,31.2500%,0.2141%,0.4253%


currently the performance of lightgbm is the best among the candidates . We will now try improving using imbalanced-learn resampling techniques and hyperparameter tuning. 
One more important point ot mention that the threshold 0.5 is not the best threshold for the model , instead 0.35 works fine but this needs more analysis and validation.

APPLY CLASS WEIGHTING

In [37]:
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scaling_factor = neg_count / pos_count
print(f"Negative class count: {neg_count}, Positive class count: {pos_count}, Scaling factor: {scaling_factor:.4f}")

Negative class count: 267519, Positive class count: 37368, Scaling factor: 7.1590


In [38]:
run_name = "run1_lgbm_class_weighting"
with mlflow.start_run(run_name=run_name) as run:
    print(f"MLflow run started with ID: {run.info.run_id}")
    mlflow.log_params(
        {
            "Model": "LightGBM",
            "Objective": "binary",
            "random_state": random_state,
            "resampling": "none",
            "class_weighting": "scale_pos_weight",
            "scale_pos_weight": float(scaling_factor),
            "split_proportions": "stratified_train: 80%, validation: 10%, test: 10%",
            "preprocessing": "one_hot_categorical_passthrough_numeric_binary",
        }
    )
    
    lgbm_class_weighted = LGBMClassifier(random_state=random_state,objective='binary', scale_pos_weight=scaling_factor)
    lgbm_class_weighted.fit(X_train_processed, y_train)

    y_val_pred_weighted = lgbm_class_weighted.predict(X_val_processed)
    y_val_proba_weighted = lgbm_class_weighted.predict_proba(X_val_processed)[:, 1]
    
    weighted_validation_metrics = metric_logger(
        y_true=y_val,
        y_pred=y_val_pred_weighted,
        y_proba=y_val_proba_weighted,
        k=10
    )
    print("Weighted validation metrics:")
    print(weighted_validation_metrics)

    mlflow.lightgbm.log_model(
        lgbm_class_weighted,
        artifact_path="model"
    )

    mlflow.sklearn.log_model(
        preprocessor,
        artifact_path="preprocessor"
    )

    print(f"MLflow run ID: {run.info.run_id}")

MLflow run started with ID: efb83cd91db2475283e4bf002ec5f8e4
[LightGBM] [Info] Number of positive: 37368, number of negative: 267519
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006171 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 870
[LightGBM] [Info] Number of data points in the train set: 304887, number of used features: 155
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.122563 -> initscore=-1.968376
[LightGBM] [Info] Start training from score -1.968376


2026/08/22 18:17:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Weighted validation metrics:
{'accuracy': 0.700952480911023, 'precision': 0.2816800830952999, 'recall': 0.9289231427959752, 'f1_score': 0.43227895392278953, 'roc_auc': 0.8564646310995565, 'precision_at_k': np.float64(0.39018630280766203), 'recall_at_k': np.float64(0.31834724898308714), 'lift_at_k': np.float64(3.183556023614388), 'pr_auc': 0.36839641116343086}


2026/08/22 18:17:16 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/08/22 18:17:23 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/08/22 18:17:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/22 18:17:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For 

MLflow run ID: efb83cd91db2475283e4bf002ec5f8e4
🏃 View run run1_lgbm_class_weighting at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/1/runs/efb83cd91db2475283e4bf002ec5f8e4
🧪 View experiment at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/1


### 6.9 Class-weighted top-K evaluation

Evaluate the class-weighted model at the same campaign capacities as the unweighted baseline. The original baseline probabilities remain stored separately.

In [39]:
baseline_proba_for_comparison = model_scores["LightGBM"]

weighted_comparison_rows = []
for model_name, model_proba in {
    "LightGBM baseline": baseline_proba_for_comparison,
    "LightGBM class weighted": y_val_proba_weighted,
}.items():
    for top_k in top_k_percentages:
        weighted_comparison_rows.append({
            "model": model_name,
            "selection_percentage": top_k,
            "precision_at_k": precision_at_k(y_val, model_proba, top_k),
            "recall_at_k": recall_at_k(y_val, model_proba, top_k),
            "lift_at_k": Lift_at_k(y_val, model_proba, top_k),
            "pr_auc": pr_auc(y_val, model_proba),
            "roc_auc": roc_auc_score(y_val, model_proba),
        })

weighted_comparison_results = pd.DataFrame(weighted_comparison_rows)
display(weighted_comparison_results.style.format({
    "precision_at_k": "{:.4%}",
    "recall_at_k": "{:.4%}",
    "lift_at_k": "{:.3f}",
    "pr_auc": "{:.4f}",
    "roc_auc": "{:.4f}",
}))

,model,selection_percentage,precision_at_k,recall_at_k,lift_at_k,pr_auc,roc_auc
0,LightGBM baseline,1,46.1942%,3.7679%,3.769,0.3649,0.8563
1,LightGBM baseline,5,41.6273%,16.9771%,3.396,0.3649,0.8563
2,LightGBM baseline,10,39.1498%,31.9418%,3.194,0.3649,0.8563
3,LightGBM baseline,20,35.2401%,57.5037%,2.875,0.3649,0.8563
4,LightGBM class weighted,1,48.0315%,3.9178%,3.919,0.3684,0.8565
5,LightGBM class weighted,5,41.1549%,16.7844%,3.358,0.3684,0.8565
6,LightGBM class weighted,10,39.0186%,31.8347%,3.184,0.3684,0.8565
7,LightGBM class weighted,20,35.4369%,57.8249%,2.891,0.3684,0.8565


### 6.10 Class-weighted threshold analysis

Class weighting can change the score distribution, so the unweighted threshold must not be reused automatically.

In [40]:
weighted_threshold_rows = []
for threshold in np.arange(0.05, 1.00, 0.05):
    weighted_threshold_pred = (y_val_proba_weighted >= threshold).astype(int)
    weighted_threshold_rows.append({
        "threshold": round(float(threshold), 2),
        "contact_rate": weighted_threshold_pred.mean(),
        "precision": precision_score(y_val, weighted_threshold_pred, zero_division=0),
        "recall": recall_score(y_val, weighted_threshold_pred, zero_division=0),
        "f1_score": f1_score(y_val, weighted_threshold_pred, zero_division=0),
    })

weighted_threshold_results = pd.DataFrame(weighted_threshold_rows)
display(weighted_threshold_results.style.format({
    "threshold": "{:.2f}",
    "contact_rate": "{:.4%}",
    "precision": "{:.4%}",
    "recall": "{:.4%}",
    "f1_score": "{:.4%}",
}))

,threshold,contact_rate,precision,recall,f1_score
0,0.05,54.6693%,22.3470%,99.6789%,36.5091%
1,0.10,53.5908%,22.7722%,99.5718%,37.0671%
2,0.15,52.1188%,23.3802%,99.4220%,37.8577%
3,0.20,50.0643%,24.2138%,98.9082%,38.9036%
4,0.25,48.1147%,25.0368%,98.2873%,39.9079%
5,0.30,46.9838%,25.5166%,97.8163%,40.4748%
6,0.35,45.9001%,25.9590%,97.2169%,40.9764%
7,0.40,44.4675%,26.5416%,96.2963%,41.6135%
8,0.45,42.4234%,27.2823%,94.4337%,42.3341%
9,0.50,40.4188%,28.1680%,92.8923%,43.2279%


### 6.11 LightGBM hyperparameter tuning with cross-validation

This experiment compares unweighted and class-weighted LightGBM using the same randomized hyperparameter search. Cross-validation is performed only inside the training split. The validation split is used only for final candidate comparison, and the final test split remains untouched.

In [41]:
from sklearn.model_selection import StratifiedKFold

cv_splitter = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=random_state,
)

lgbm_param_distributions = {
    "n_estimators": [200, 400, 700],
    "learning_rate": [0.03, 0.05, 0.10],
    "num_leaves": [15, 31, 63],
    "max_depth": [-1, 5, 8, 12],
    "min_child_samples": [20, 50, 100],
    "subsample": [0.8, 1.0],
    "subsample_freq": [1],
    "colsample_bytree": [0.8, 1.0],
    "reg_alpha": [0.0, 0.1, 1.0],
    "reg_lambda": [0.0, 0.1, 1.0],
}

lgbm_variants = {
    "unweighted": None,
    "class_weighted": float(scaling_factor),
}

tuned_models = {}
tuned_validation_rows = []

with mlflow.start_run(run_name="tier1_lgbm_cv_comparison") as parent_run:
    mlflow.log_params({
        "experiment_stage": "tier1_lgbm_hyperparameter_tuning",
        "cv_strategy": "stratified_3_fold_on_training_only",
        "search_scoring": "average_precision",
        "n_iter": 12,
        "random_state": random_state,
        "validation_split": "untouched_during_cv",
        "test_split": "untouched",
    })

    for variant_name, variant_scale_pos_weight in lgbm_variants.items():
        with mlflow.start_run(
            run_name=f"tier1_lgbm_{variant_name}",
            nested=True,
        ) as child_run:
            estimator_params = {
                "objective": "binary",
                "random_state": random_state,
                "n_jobs": -1,
                "verbosity": -1,
            }
            if variant_scale_pos_weight is not None:
                estimator_params["scale_pos_weight"] = variant_scale_pos_weight

            search = RandomizedSearchCV(
                estimator=LGBMClassifier(**estimator_params),
                param_distributions=lgbm_param_distributions,
                n_iter=12,
                scoring="average_precision",
                cv=cv_splitter,
                refit=True,
                random_state=random_state,
                n_jobs=-1,
                verbose=0,
                return_train_score=False,
            )
            search.fit(X_train_processed, y_train)

            best_model = search.best_estimator_
            tuned_models[variant_name] = best_model
            tuned_proba = best_model.predict_proba(X_val_processed)[:, 1]
            tuned_pred = best_model.predict(X_val_processed)

            mlflow.log_params({
                "variant": variant_name,
                "class_weighting": "scale_pos_weight" if variant_scale_pos_weight is not None else "none",
                "scale_pos_weight": str(variant_scale_pos_weight),
            })
            mlflow.log_params({
                f"best_{key}": str(value)
                for key, value in search.best_params_.items()
            })
            mlflow.log_metric("best_cv_average_precision", float(search.best_score_))

            mlflow.log_metrics({
                "validation_accuracy_at_0_5": float(accuracy_score(y_val, tuned_pred)),
                "validation_precision_at_0_5": float(precision_score(y_val, tuned_pred, zero_division=0)),
                "validation_recall_at_0_5": float(recall_score(y_val, tuned_pred, zero_division=0)),
                "validation_f1_at_0_5": float(f1_score(y_val, tuned_pred, zero_division=0)),
                "validation_pr_auc": float(pr_auc(y_val, tuned_proba)),
                "validation_roc_auc": float(roc_auc_score(y_val, tuned_proba)),
            })

            for top_k in top_k_percentages:
                precision_value = float(precision_at_k(y_val, tuned_proba, top_k))
                recall_value = float(recall_at_k(y_val, tuned_proba, top_k))
                lift_value = float(Lift_at_k(y_val, tuned_proba, top_k))
                mlflow.log_metrics({
                    f"validation_precision_at_{top_k}pct": precision_value,
                    f"validation_recall_at_{top_k}pct": recall_value,
                    f"validation_lift_at_{top_k}pct": lift_value,
                })
                tuned_validation_rows.append({
                    "variant": variant_name,
                    "selection_percentage": top_k,
                    "precision_at_k": precision_value,
                    "recall_at_k": recall_value,
                    "lift_at_k": lift_value,
                    "pr_auc": float(pr_auc(y_val, tuned_proba)),
                    "roc_auc": float(roc_auc_score(y_val, tuned_proba)),
                })

            cv_results_for_logging = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
            mlflow.log_text(
                cv_results_for_logging.to_csv(index=False),
                artifact_file="cv_results.csv",
            )
            mlflow.lightgbm.log_model(best_model, artifact_path="model")
            print(f"Completed {variant_name}; MLflow run ID: {child_run.info.run_id}")

    tuned_validation_results = pd.DataFrame(tuned_validation_rows)
    mlflow.log_text(
        tuned_validation_results.to_csv(index=False),
        artifact_file="tuned_validation_results.csv",
    )

display(tuned_validation_results.style.format({
    "precision_at_k": "{:.4%}",
    "recall_at_k": "{:.4%}",
    "lift_at_k": "{:.3f}",
    "pr_auc": "{:.4f}",
    "roc_auc": "{:.4f}",
}))

2026/08/22 18:21:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/22 18:21:24 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/08/22 18:21:41 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Completed unweighted; MLflow run ID: 5fddf58edd1247739d685ee41c9f5cb8
🏃 View run tier1_lgbm_unweighted at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/1/runs/5fddf58edd1247739d685ee41c9f5cb8
🧪 View experiment at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/1


2026/08/22 18:25:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/22 18:25:39 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/08/22 18:25:51 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Completed class_weighted; MLflow run ID: 543aea08aecd43ee89e5ca66dbf57816
🏃 View run tier1_lgbm_class_weighted at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/1/runs/543aea08aecd43ee89e5ca66dbf57816
🧪 View experiment at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/1
🏃 View run tier1_lgbm_cv_comparison at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/1/runs/46d9a5e2e9df498f9dcd12bec5a29927
🧪 View experiment at: https://dagshub.com/Youngboy01/vehicle-Insurance.mlflow/#/experiments/1


,variant,selection_percentage,precision_at_k,recall_at_k,lift_at_k,pr_auc,roc_auc
0,unweighted,1,50.3937%,4.1105%,4.112,0.3689,0.8571
1,unweighted,5,41.4173%,16.8915%,3.379,0.3689,0.8571
2,unweighted,10,39.2548%,32.0274%,3.203,0.3689,0.8571
3,unweighted,20,35.5681%,58.0390%,2.902,0.3689,0.8571
4,class_weighted,1,50.6562%,4.1319%,4.133,0.3700,0.8573
5,class_weighted,5,41.4698%,16.9129%,3.384,0.3700,0.8573
6,class_weighted,10,39.6484%,32.3485%,3.235,0.3700,0.8573
7,class_weighted,20,35.5025%,57.9319%,2.897,0.3700,0.8573
